# DynamicORB — Opening Range Breakout Research

This notebook asks one question: **can an Opening Range Breakout (ORB) strategy be improved by choosing its
parameters, or its opening-range timeframe, from market context, out of sample?**

Universe: 10 ETFs (CPER, DIA, GLD, IEF, IWM, QQQ, SHY, SLV, SPY, TLT), 1-minute regular-session bars,
2003–2026. **Benchmark: SPY buy & hold** (dividend-adjusted, so a total-return proxy).

| Experiment | Question |
|---|---|
| **A** | Does any ORB variant from a 2,400-configuration search survive an honest in-sample → out-of-sample protocol? |
| **B** | Can a walk-forward Random Forest pick the best OR window (5/15/30/45/60 min) per ETF each morning better than simple baselines? |

### Notebook layout
1. **Setup & configuration**: every tunable lives in one place.
2. **Data layer**: loading, cleaning, caching, data-quality report.
3. **Session panels & causal features**: every feature is computable *before* it is used.
4. **Execution engine**: vectorised, conservative fill model.
5. **Portfolio simulator & metrics**: integer sizing, gross-leverage cap, daily mark-to-market, Deflated Sharpe.
6. **Self-tests**: synthetic sessions that pin the fill rules and the no-look-ahead guarantees.
7. **Search protocol**, then **Experiments A and B**.
8. **Roadmap**: planned modules not yet built.

### Defects fixed relative to the original version of this notebook
These were found in an audit. Several of them inflated the results it reported. NQ futures have since been
removed from scope, so NQ-specific defects are not listed.

| # | Area | Defect | Effect | Fix |
|---|---|---|---|---|
| 1 | Look-ahead | `sma_trend` direction compared **today's close** with an SMA that included today. | It dominated every top configuration. | The trend uses the prior close vs the prior SMA20. |
| 2 | Look-ahead | `nr7`, `avgvol14`, `bb_width`, `rv5/rv20`, `atr_ratio`, `hurst20` all included the current day. | Leaked the day's outcome into filters and ML features. | Every feature is `shift(1)`-ed or built only from data before 09:30. A self-test enforces this. |
| 3 | Look-ahead | ML evaluation fell back to the *best available* timeframe whenever the predicted one did not trade, and trained only on days where ≥ 2 timeframes traded. | The "feature-selected" curve partly used oracle choices. | The policy commits before the open. A timeframe that does not trigger earns 0 R, and every day is used. |
| 4 | Selection bias | The "best" configuration was picked by **OOS** Sharpe, and the full-sample result was then presented. | The OOS result was no longer out of sample. | Selection uses in-sample data only. The OOS result of that single choice is reported, plus a Deflated Sharpe Ratio for the number of trials. |
| 5 | Sizing | Equity was updated mid-day from trades that had not closed yet, and the leverage cap applied per trade rather than to the whole book. | Mild look-ahead and understated leverage. | Sizing uses start-of-day equity. The leverage cap covers all concurrently open positions. |
| 6 | Fills | Target won same-bar ties. Targets filled on a touch. The entry bar was never checked for a stop. VWAP and breakout volume were read from the same bar a stop order filled on. EOD exits had no slippage. | Optimistic fills. | Stop wins ties. Targets need a one-tick trade-through. A stop entry is checked for a stop on its own bar. Stop-entry filters only use information from earlier bars. Every market exit pays slippage. |
| 7 | Metrics | Sharpe was computed over **trade days only** and annualised with √252. | Inflated Sharpe. | Daily mark-to-market over every session. Trade-free days count as 0 %. |
| 8 | ETF data | From **2022-03-07** the ETF files switch from `pitrading` (consolidated volume) to `iex` (≈ 2 % of volume, sparse bars for small ETFs). | The absolute volume filter silently removed most OOS trades, and relative volume broke at the seam. | Relative volume is computed within each data-source regime. Sessions are gated by bar coverage. The absolute share-volume filter is removed. |
| 9 | Filters | `MIN_ATR = 0.50` (a stock-paper filter) permanently excluded low-priced, low-volatility ETFs such as SHY, while 0.10 × ATR stops could fall to a single tick. | Arbitrary universe and untradeable stops. | Replaced by a minimum stop distance of 4 ticks. |
| 10 | Hurst | The Hurst exponent was estimated on *returns* rather than on the log-price path. | The feature measured nothing useful. | Variance-of-lagged-differences estimator on log price. |
| 11 | Hygiene | Duplicated cells, re-imports, `eval()` on strings, bare `except: continue`. | Hard to maintain, and errors were swallowed. | One shared engine, typed configs, explicit validation. |

## 1. Setup & configuration

In [1]:
from __future__ import annotations

import hashlib
import itertools
import json
import logging
import math
import os
import time
import warnings
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path
from typing import Iterable, Literal

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from scipy import stats
from sklearn.ensemble import RandomForestRegressor

# nan-reductions over all-missing OR windows are expected and handled explicitly.
warnings.filterwarnings("ignore", category=RuntimeWarning, message="All-NaN slice encountered")
warnings.filterwarnings("ignore", category=RuntimeWarning, message="Mean of empty slice")
warnings.filterwarnings("ignore", category=RuntimeWarning, message="invalid value encountered")
warnings.filterwarnings("ignore", category=RuntimeWarning, message="divide by zero encountered")

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-7s %(message)s",
                    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("dynamicorb")
pio.renderers.default = "plotly_mimetype+notebook_connected"   # keeps the .ipynb small
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_ROOT = Path("/Users/andrew/main/trading/data")
ETF_DIR = DATA_ROOT / "etfs"
ETF_SYMBOLS: tuple[str, ...] = ("CPER", "DIA", "GLD", "IEF", "IWM", "QQQ", "SHY", "SLV", "SPY", "TLT")
BENCHMARK = "SPY"
CACHE_DIR = Path("data_cache")          # parquet cache of cleaned bars (git-ignored)
TZ = "America/New_York"                 # all timestamps are converted to ET wall-clock

# ── Session geometry (1-minute bars labelled by bar-open time) ───────────────
RTH_OPEN_MIN = 9 * 60 + 30              # 09:30 → column 0
RTH_BARS = 390                          # 09:30 … 15:59
SEED = 42

# Set QUICK_RUN = True for a fast smoke test (small search, fewer trees).
QUICK_RUN = os.environ.get("DORB_QUICK_RUN", "0") == "1"


@dataclass(frozen=True)
class CleaningConfig:
    """Data-quality thresholds. Changing any value invalidates the cache."""
    spike_window: int = 15                  # centred rolling-median window (bars)
    spike_threshold: float = 0.05           # |log(px / median)| above this = bad print
    min_session_coverage: float = 0.80      # share of minutes present up to the last bar
    min_or_coverage: float = 0.80           # share of minutes present inside the OR window
    min_last_col: int = 209                 # session must run to ≥ 12:59 (keeps half-days)
    max_open_delay: int = 2                 # first bar must open by 09:32 (≈10 % of ETF sessions lack 09:30)


@dataclass(frozen=True)
class InstrumentSpec:
    """Execution economics for one tradable instrument."""
    symbol: str
    tick_size: float
    point_value: float          # $ P&L per 1.0 price move per unit (1.0 for shares)
    commission: float           # $ per unit per side
    slippage_ticks: float = 1.0  # adverse ticks on market / stop fills
    min_price: float = 0.0      # skip sessions whose previous close is below this
    integer_units: bool = True


@dataclass(frozen=True)
class FeatureConfig:
    atr_len: int = 14
    atr_slow_len: int = 50
    sma_len: int = 20
    rel_vol_lookback: int = 14
    rel_vol_min_periods: int = 10
    breakout_vol_bars: int = 20
    hurst_window: int = 64
    hurst_max_lag: int = 16


@dataclass(frozen=True)
class ExecutionConfig:
    min_stop_ticks: int = 4         # stops tighter than this are not tradable
    exit_col: int = RTH_BARS - 1    # flatten at the close of the 15:59 bar (16:00)


@dataclass(frozen=True)
class PortfolioConfig:
    initial_equity: float = 25_000.0
    risk_per_trade: float = 0.01        # fraction of start-of-day equity risked per trade
    max_gross_leverage: float = 4.0     # cap on concurrently open notional / equity


CLEAN = CleaningConfig()
FEAT = FeatureConfig()
EXEC = ExecutionConfig()
PORT = PortfolioConfig()

# The commission is a typical retail per-share rate. Verify it against your broker's current schedule.
ETF_SPEC_DEFAULTS = dict(tick_size=0.01, point_value=1.0, commission=0.0035, slippage_ticks=1.0, min_price=5.0)

log.info("QUICK_RUN=%s  cache=%s", QUICK_RUN, CACHE_DIR.resolve())

22:25:36  INFO    QUICK_RUN=False  cache=/Users/andrew/main/programming/projects/DynamicORB-system/data_cache


## 2. Data layer

**Cleaning rules**, applied to every ETF:
1. Duplicate timestamps — keep the last row.
2. Non-finite or non-positive prices, and negative volume — drop the bar.
3. Inconsistent OHLC (`high < max(open, close)` or `low > min(open, close)`) — drop the bar.
4. Bad prints — any of O/H/L/C more than 5 % (log) away from a centred 15-bar rolling median of closes. A real
   move persists past half the window and moves the median with it, so only isolated prints get flagged.
5. Keep regular trading hours only (09:30–15:59 bar opens, ET wall-clock).

**Known data issue.** On **2022-03-07** every file switches from `pitrading` (consolidated volume) to `iex`
(single venue, about 2 % of volume). Volume levels are not comparable across that seam, and smaller ETFs
become sparse. Volume features are therefore computed within each source regime, and the coverage gate in
§3 drops sparse sessions.

Cleaned frames are cached to parquet. The cache key covers the source file's size and mtime and the cleaning
config.

In [2]:
OHLC = ["open", "high", "low", "close"]
CACHE_VERSION = 3


def _cache_key(name: str, src: Path) -> str:
    st = src.stat()
    raw = f"{src.resolve()}|{st.st_size}|{st.st_mtime_ns}|{CACHE_VERSION}|{CLEAN}"
    return f"{name}-{hashlib.sha1(raw.encode()).hexdigest()[:12]}"


def _cache_read(key: str) -> tuple[dict[str, pd.DataFrame], dict] | None:
    meta = CACHE_DIR / f"{key}.json"
    if not meta.exists():
        return None
    info = json.loads(meta.read_text())
    frames = {name: pd.read_parquet(CACHE_DIR / f"{key}.{name}.parquet") for name in info["frames"]}
    return frames, info["report"]


def _cache_write(key: str, frames: dict[str, pd.DataFrame], report: dict) -> None:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    for name, df in frames.items():
        df.to_parquet(CACHE_DIR / f"{key}.{name}.parquet")
    (CACHE_DIR / f"{key}.json").write_text(json.dumps({"frames": list(frames), "report": report}))


def clean_bars(df: pd.DataFrame, cfg: CleaningConfig = CLEAN) -> tuple[pd.DataFrame, dict[str, int]]:
    """Apply the cleaning rules to a DatetimeIndexed OHLCV frame. Returns (clean, counts)."""
    rep: dict[str, int] = {"raw_bars": int(len(df))}
    df = df.sort_index(kind="stable")

    dup = df.index.duplicated(keep="last")
    rep["duplicate_ts"] = int(dup.sum())
    df = df.loc[~dup]

    px = df[OHLC].to_numpy(dtype=np.float64)
    vol = df["volume"].to_numpy(dtype=np.float64)
    finite = np.isfinite(px).all(axis=1) & (px > 0).all(axis=1) & np.isfinite(vol) & (vol >= 0)
    rep["nonfinite_or_nonpositive"] = int((~finite).sum())
    df, px = df.loc[finite], px[finite]

    o, h, l, c = px.T
    eps = 1e-9
    consistent = (h + eps >= np.maximum(o, c)) & (l - eps <= np.minimum(o, c))
    rep["inconsistent_ohlc"] = int((~consistent).sum())
    df, px = df.loc[consistent], px[consistent]

    med = (df["close"].rolling(cfg.spike_window, center=True, min_periods=cfg.spike_window // 2 + 1)
           .median().to_numpy())
    dev = np.abs(np.log(px / med[:, None])).max(axis=1)
    spike = np.nan_to_num(dev, nan=0.0) > cfg.spike_threshold
    rep["spikes"] = int(spike.sum())
    df = df.loc[~spike]

    minute = df.index.hour * 60 + df.index.minute
    rth = (minute >= RTH_OPEN_MIN) & (minute < RTH_OPEN_MIN + RTH_BARS)
    rep["clean_bars"] = int(len(df))
    rep["clean_rth_bars"] = int(rth.sum())
    return df, rep


def load_etf(symbol: str) -> tuple[pd.DataFrame, dict]:
    """Load one ETF 1-minute file → cleaned RTH bars (ET wall-clock index) + quality report."""
    src = ETF_DIR / f"{symbol}_clean_1min.csv"
    if not src.exists():
        raise FileNotFoundError(src)
    key = _cache_key(f"etf-{symbol}", src)
    cached = _cache_read(key)
    if cached is not None:
        return cached[0]["rth"], cached[1]

    raw = pd.read_csv(src, engine="pyarrow")
    raw.columns = raw.columns.str.lower()
    missing = {"datetime", *OHLC, "volume"} - set(raw.columns)
    if missing:
        raise ValueError(f"{src.name}: missing columns {sorted(missing)}")
    ts = pd.DatetimeIndex(pd.to_datetime(raw["datetime"], format="%Y-%m-%d %H:%M:%S"))
    if ts.tz is not None:                                   # defensive: normalise to ET wall-clock
        ts = ts.tz_convert(TZ).tz_localize(None)
    cols = [*OHLC, "volume"] + (["source"] if "source" in raw.columns else [])
    df = raw[cols].set_index(ts)
    df.index.name = "ts"
    if "source" not in df.columns:
        df["source"] = "unknown"
    df["source"] = df["source"].astype(str)

    df, rep = clean_bars(df)
    minute = df.index.hour * 60 + df.index.minute
    df = df.loc[(minute >= RTH_OPEN_MIN) & (minute < RTH_OPEN_MIN + RTH_BARS)]
    _cache_write(key, {"rth": df}, rep)
    return df, rep

### 2.1 Load everything

The first run parses about 18 M rows (roughly a minute). Later runs read the parquet cache.

In [3]:
t0 = time.perf_counter()
etf_bars: dict[str, pd.DataFrame] = {}
quality_rows: list[dict] = []
for sym in ETF_SYMBOLS:
    try:
        bars, rep = load_etf(sym)
    except (FileNotFoundError, ValueError) as exc:
        log.warning("%s skipped: %s", sym, exc)
        continue
    etf_bars[sym] = bars
    quality_rows.append({"asset": sym, **rep})

if BENCHMARK not in etf_bars:
    raise RuntimeError(f"Benchmark {BENCHMARK} failed to load.")
log.info("Loaded %d ETFs in %.1fs", len(etf_bars), time.perf_counter() - t0)

22:25:36  INFO    Loaded 10 ETFs in 0.8s


## 3. Session panels & causal features

Each instrument becomes a **session panel**: one `(n_sessions × 390)` float32 matrix per field, where column
*j* is the bar opening at 09:30 + *j* minutes and a missing bar is `NaN`. Every OR window, breakout scan and
exit scan then becomes a vectorised operation over all sessions at once.

**Feature timing contract.** A daily feature for date *d* may only use information available **before 09:30
on *d*** (the 09:30 open counts as known for gap features). OR features for window *k* are known at
09:30 + *k*. The look-ahead self-test in §6 enforces this.

In [4]:
@dataclass
class SessionPanel:
    dates: pd.DatetimeIndex
    open: np.ndarray
    high: np.ndarray
    low: np.ndarray
    close: np.ndarray
    volume: np.ndarray
    vwap: np.ndarray            # cumulative session VWAP through bar j (inclusive)
    vol_avg_prev: np.ndarray    # mean volume of the previous N bars (exclusive of bar j)
    first_col: np.ndarray       # first present bar (-1 if none)
    last_col: np.ndarray        # last present bar ≤ exit_col (-1 if none)
    coverage: np.ndarray        # share of minutes present in [0, last_col]

    @property
    def n(self) -> int:
        return len(self.dates)


def build_panel(bars: pd.DataFrame, feat: FeatureConfig = FEAT, exec_cfg: ExecutionConfig = EXEC) -> SessionPanel:
    if bars.empty:
        raise ValueError("Cannot build a panel from an empty frame.")
    idx = bars.index
    codes, dates = pd.factorize(idx.normalize(), sort=True)
    col = (idx.hour * 60 + idx.minute - RTH_OPEN_MIN).to_numpy()
    if col.min() < 0 or col.max() >= RTH_BARS:
        raise ValueError("Bars outside RTH reached build_panel.")
    n = len(dates)

    def mat(values: np.ndarray) -> np.ndarray:
        m = np.full((n, RTH_BARS), np.nan, dtype=np.float32)
        m[codes, col] = values
        return m

    o, h, l, c = (mat(bars[k].to_numpy(np.float64)) for k in OHLC)
    v = mat(bars["volume"].to_numpy(np.float64))
    vol_avg_prev = mat(bars["volume"].rolling(feat.breakout_vol_bars, min_periods=feat.breakout_vol_bars)
                       .mean().shift(1).to_numpy())

    present = ~np.isnan(c)
    present[:, exec_cfg.exit_col + 1:] = False
    any_bar = present.any(axis=1)
    first_col = np.where(any_bar, present.argmax(axis=1), -1)
    last_col = np.where(any_bar, RTH_BARS - 1 - present[:, ::-1].argmax(axis=1), -1)
    coverage = np.where(any_bar, present.sum(axis=1) / np.maximum(last_col + 1, 1), 0.0)

    tp = (h + l + c) / 3.0
    cum_pv = np.cumsum(np.nan_to_num(tp * v), axis=1)
    cum_v = np.cumsum(np.nan_to_num(v), axis=1)
    vwap = np.where(cum_v > 0, cum_pv / np.where(cum_v > 0, cum_v, 1.0), np.nan).astype(np.float32)

    return SessionPanel(pd.DatetimeIndex(dates, name="date"), o, h, l, c, v, vwap, vol_avg_prev,
                        first_col.astype(np.int32), last_col.astype(np.int32), coverage)


def daily_from_panel(p: SessionPanel) -> pd.DataFrame:
    rows = np.arange(p.n)
    fc, lc = np.maximum(p.first_col, 0), np.maximum(p.last_col, 0)
    ok = p.last_col >= 0
    d = pd.DataFrame({
        "open": np.where(ok, p.open[rows, fc], np.nan),
        "high": np.nanmax(p.high, axis=1),
        "low": np.nanmin(p.low, axis=1),
        "close": np.where(ok, p.close[rows, lc], np.nan),
        "volume": np.nansum(p.volume, axis=1),
        "first_col": p.first_col,
    }, index=p.dates).astype({"open": float, "high": float, "low": float, "close": float, "volume": float})
    d[OHLC] = d[OHLC].round(6)                          # strip float32 representation noise
    return d


def wilder(x: pd.Series, n: int) -> pd.Series:
    return x.ewm(alpha=1.0 / n, adjust=False, min_periods=n).mean()


def rolling_hurst(log_px: pd.Series, window: int, max_lag: int) -> pd.Series:
    """Rolling Hurst exponent from the scaling of lagged log-price differences: std(Δ_lag) ∝ lag^H."""
    lags = np.arange(2, max_lag + 1)
    cols = []
    for lag in lags:
        w = window - lag
        cols.append(np.log(log_px.diff(lag).rolling(w, min_periods=w).std()).to_numpy())
    y = np.column_stack(cols)
    y[~np.isfinite(y)] = np.nan
    x = np.log(lags)
    xc = x - x.mean()
    slope = ((y - y.mean(axis=1, keepdims=True)) @ xc) / (xc @ xc)
    return pd.Series(slope, index=log_px.index)


FEATURE_COLUMNS = ["gap_atr", "gap_pct", "atr_ratio_prev", "rv20_prev", "rv_ratio_prev", "prior_range_atr",
                   "prior_close_loc", "trend_z_prev", "hurst_prev", "nr7_prev", "dow"]


def build_daily_features(daily: pd.DataFrame, feat: FeatureConfig = FEAT) -> pd.DataFrame:
    """
    Causal daily features indexed by session date. `daily` needs open/high/low/close/prev_close.
    Each column for date d only uses data before 09:30 on d (plus the 09:30 open for the gap columns).
    """
    o, h, l, c, pc = (daily[k] for k in ("open", "high", "low", "close", "prev_close"))
    f = pd.DataFrame(index=daily.index)

    tr = pd.concat([h - l, (h - pc).abs(), (l - pc).abs()], axis=1).max(axis=1, skipna=True)
    atr = wilder(tr, feat.atr_len)
    atr_slow = wilder(tr, feat.atr_slow_len)
    f["atr_prev"] = atr.shift(1)

    ret = np.log(c / pc)                                     # daily log return
    log_idx = ret.fillna(0.0).cumsum()                       # log price index (robust to missing sessions)
    idx_px = np.exp(log_idx)
    sma = idx_px.rolling(feat.sma_len, min_periods=feat.sma_len).mean()
    trend_up = (idx_px > sma).astype(float).where(sma.notna())
    rv20 = ret.rolling(20, min_periods=20).std()
    rv5 = ret.rolling(5, min_periods=5).std()
    f["trend_up_prev"] = trend_up.shift(1)
    f["trend_z_prev"] = (np.log(idx_px / sma) / rv20).shift(1)

    rng = h - l
    nr7 = (rng <= rng.rolling(7, min_periods=7).min()).astype(float).where(rng.rolling(7, min_periods=7).min().notna())
    f["nr7_prev"] = nr7.shift(1)
    f["prev_close"] = pc
    f["gap"] = o - pc
    f["gap_atr"] = f["gap"] / f["atr_prev"]
    f["gap_pct"] = f["gap"] / pc
    f["rv20_prev"] = (rv20 * np.sqrt(252)).shift(1)
    f["rv_ratio_prev"] = (rv5 / rv20).shift(1)
    f["atr_ratio_prev"] = (atr / atr_slow).shift(1)
    f["prior_range_atr"] = rng.shift(1) / f["atr_prev"]
    f["prior_close_loc"] = ((c - l) / rng.where(rng > 0)).shift(1)
    f["hurst_prev"] = rolling_hurst(log_idx, feat.hurst_window, feat.hurst_max_lag).shift(1)
    f["dow"] = daily.index.dayofweek.astype(float)
    return f.replace([np.inf, -np.inf], np.nan)


def build_or_stats(p: SessionPanel, k: int, session_ok: np.ndarray, vol_regime: np.ndarray,
                   clean: CleaningConfig = CLEAN, feat: FeatureConfig = FEAT) -> pd.DataFrame:
    """Opening-range statistics for a k-minute window. Everything here is known at 09:30 + k."""
    rows = np.arange(p.n)
    c_k = p.close[:, :k]
    present = ~np.isnan(c_k)
    last_in = k - 1 - present[:, ::-1].argmax(axis=1)
    valid = session_ok & (present.mean(axis=1) >= clean.min_or_coverage) & (p.last_col > k)
    or_open = p.open[rows, np.maximum(p.first_col, 0)].astype(np.float64)    # first bar ≤ 09:32 (gated)
    or_close = c_k[rows, last_in].astype(np.float64)
    or_high = np.nanmax(p.high[:, :k], axis=1).astype(np.float64)
    or_low = np.nanmin(p.low[:, :k], axis=1).astype(np.float64)
    or_vol = np.nansum(p.volume[:, :k], axis=1).astype(np.float64)
    direction = np.sign(or_close - or_open).astype(np.int8)
    direction[~valid] = 0

    df = pd.DataFrame({"valid": valid, "or_open": or_open, "or_close": or_close, "or_high": or_high,
                       "or_low": or_low, "or_range": or_high - or_low, "or_vol": or_vol,
                       "direction": direction}, index=p.dates)
    # Relative volume vs the previous N sessions *within the same data-source regime*.
    v = df["or_vol"].where(valid)
    base = v.groupby(vol_regime).transform(
        lambda s: s.rolling(feat.rel_vol_lookback, min_periods=feat.rel_vol_min_periods).mean().shift(1))
    df["rel_vol"] = (v / base).where(base > 0)
    return df


OR_WINDOWS: tuple[int, ...] = (5, 15, 30, 45, 60)


@dataclass
class AssetData:
    spec: InstrumentSpec
    panel: SessionPanel
    daily: pd.DataFrame
    features: pd.DataFrame
    session_ok: np.ndarray
    or_stats: dict[int, pd.DataFrame]
    quality: dict = field(default_factory=dict)


def build_asset(spec: InstrumentSpec, bars: pd.DataFrame, *, vol_regime: pd.Series | None = None,
                clean: CleaningConfig = CLEAN) -> AssetData:
    panel = build_panel(bars)
    daily = daily_from_panel(panel)
    daily["prev_close"] = daily["close"].shift(1)
    features = build_daily_features(daily)

    session_ok = ((panel.coverage >= clean.min_session_coverage) & (panel.last_col >= clean.min_last_col)
                  & (panel.first_col >= 0) & (panel.first_col <= clean.max_open_delay))
    regime = (vol_regime.reindex(panel.dates).fillna("unknown").to_numpy() if vol_regime is not None
              else np.zeros(panel.n, dtype=np.int8))
    or_stats = {k: build_or_stats(panel, k, session_ok, regime) for k in OR_WINDOWS}
    quality = {"sessions": panel.n, "tradable_sessions": int(session_ok.sum()),
               "median_coverage": float(np.median(panel.coverage)),
               "first": panel.dates[0].date(), "last": panel.dates[-1].date()}
    return AssetData(spec, panel, daily, features, session_ok, or_stats, quality)

In [5]:
t0 = time.perf_counter()
assets: dict[str, AssetData] = {}
for sym, bars in etf_bars.items():
    regime = bars["source"].groupby(bars.index.normalize()).first()
    spec = InstrumentSpec(symbol=sym, **ETF_SPEC_DEFAULTS)
    assets[sym] = build_asset(spec, bars, vol_regime=regime)
log.info("Built %d asset panels in %.1fs", len(assets), time.perf_counter() - t0)

quality = pd.DataFrame(quality_rows).set_index("asset")
quality = quality.join(pd.DataFrame({k: a.quality for k, a in assets.items()}).T)
quality["tradable_%"] = 100 * quality["tradable_sessions"] / quality["sessions"]
print("Data-quality report (bar counts after each cleaning rule; sessions after the coverage gate):")
display(quality[["first", "last", "raw_bars", "duplicate_ts", "nonfinite_or_nonpositive", "inconsistent_ohlc",
                 "spikes", "clean_rth_bars", "sessions", "tradable_sessions", "tradable_%", "median_coverage"]])

22:25:39  INFO    Built 10 asset panels in 2.8s


Data-quality report (bar counts after each cleaning rule; sessions after the coverage gate):


,first,last,raw_bars,duplicate_ts,nonfinite_or_nonpositive,inconsistent_ohlc,spikes,clean_rth_bars,sessions,tradable_sessions,tradable_%,median_coverage
asset,,,,,,,,,,,,
CPER,2011-11-15,2026-09-21,109059,0,0,0,7,109052,3313,6,0.1811,0.0310
DIA,2002-12-30,2026-09-21,2067170,0,0,0,0,2067170,5966,4780,80.1207,0.9615
GLD,2004-11-18,2026-09-21,1924936,0,0,0,0,1924936,5490,4440,80.8743,0.9615
IEF,2002-12-30,2026-09-21,1560831,0,0,0,0,1560831,5966,2198,36.8421,0.7150
IWM,2002-12-30,2026-09-21,2217669,0,0,0,1,2217668,5966,5423,90.8984,0.9692
QQQ,2002-12-30,2026-09-21,2245015,0,0,0,4,2245011,5968,5615,94.0851,0.9718
SHY,2002-12-30,2026-09-21,1523666,0,0,0,0,1523666,5966,2536,42.5075,0.7564
SLV,2006-04-28,2026-09-21,1752691,0,0,0,0,1752691,5128,3826,74.6100,0.9641
SPY,2002-12-30,2026-09-21,2250959,0,0,0,1,2250958,5968,5678,95.1408,0.9744


## 4. Execution engine

One `TradeLogic` describes how a single session is traded. A `DayFilter` decides which sessions are traded at
all, using only information known at the end of the OR window. The engine evaluates one `TradeLogic` across
every session of an asset at once.

**Fill model (deliberately conservative)**

| Event | Rule |
|---|---|
| Direction | Sign of the OR candle (`or_candle`), optionally required to agree with the gap (`gap`) or the prior-day trend (`trend`). Doji OR candle → no trade. |
| `stop` entry | Resting stop at the OR high/low from 09:30 + k. Fill = `max(trigger, bar open)` + 1 tick slippage, which also covers a gap through the level. |
| `close` entry | First bar that **closes** beyond the level. Fill = that close + 1 tick. |
| `retest` entry | After the first touch, wait for a later bar that trades back to the level and closes beyond it. Fill = that close + 1 tick. |
| VWAP / breakout-volume filters | Evaluated on the confirmation bar for `close` and `retest`. A `stop` entry may only use the VWAP of the **previous** bar, and the volume filter is invalid with `stop` entries because that bar's volume is unknown at fill time. |
| Stop | `stop_param × ATR(prev)` or `× OR range`, rounded away from entry to the tick grid. A stop entry is checked for a stop on its own entry bar. A gap through the stop fills at the bar open. 1 tick slippage. |
| Target | Limit order. It fills only when price trades **one tick through**, and a gap through fills at the open. No slippage. |
| Same-bar stop & target | **Stop wins.** |
| End of day | Close of the last bar at or before 15:59, less 1 tick slippage. |
| Costs | Per-unit commission on both sides plus slippage, in instrument units (`R_net` is after costs). |

In [6]:
DirectionSource = Literal["or_candle", "gap", "trend"]
EntryType = Literal["stop", "close", "retest"]
StopType = Literal["atr", "or_range"]
TargetType = Literal["eod", "or_mult", "r_mult"]


@dataclass(frozen=True)
class TradeLogic:
    orb_min: int = 5
    direction_source: DirectionSource = "or_candle"
    entry_type: EntryType = "stop"
    breakout_vol_mult: float = 0.0
    use_vwap_filter: bool = False
    stop_type: StopType = "atr"
    stop_param: float = 0.10
    target_type: TargetType = "eod"
    target_param: float = 0.0

    def validate(self) -> None:
        if self.orb_min not in OR_WINDOWS:
            raise ValueError(f"orb_min must be one of {OR_WINDOWS}")
        if self.direction_source not in ("or_candle", "gap", "trend"):
            raise ValueError(self.direction_source)
        if self.entry_type not in ("stop", "close", "retest"):
            raise ValueError(self.entry_type)
        if self.entry_type == "stop" and self.breakout_vol_mult > 0:
            raise ValueError("breakout-volume filter needs a bar-close entry (look-ahead with stop orders)")
        if self.stop_type not in ("atr", "or_range") or self.stop_param <= 0:
            raise ValueError("bad stop")
        if self.target_type not in ("eod", "or_mult", "r_mult"):
            raise ValueError(self.target_type)
        if self.target_type != "eod" and self.target_param <= 0:
            raise ValueError("target_param must be > 0 for price targets")


@dataclass(frozen=True)
class DayFilter:
    or_atr_min: float = 0.0
    or_atr_max: float = math.inf
    require_nr7: bool = False
    rel_vol_min: float = 0.0
    top_n: int | None = None            # cross-sectional cap ranked by relative volume


def _first_true(mask: np.ndarray) -> np.ndarray:
    """Index of the first True per row, -1 when the row has none."""
    idx = mask.argmax(axis=1)
    hit = mask[np.arange(mask.shape[0]), idx]
    return np.where(hit, idx, -1)


def _take(m: np.ndarray, rows: np.ndarray, cols: np.ndarray) -> np.ndarray:
    return m[rows, np.clip(cols, 0, m.shape[1] - 1)].astype(np.float64)


def _snap(x: np.ndarray, tick: float) -> np.ndarray:
    """Snap float32-derived prices back onto the tick grid."""
    return np.round(np.asarray(x, dtype=np.float64) / tick) * tick


CANDIDATE_COLUMNS = ["asset", "date", "direction", "or_high", "or_low", "or_range", "or_atr", "atr", "rel_vol",
                     "nr7_prev", "triggered", "entry_col", "exit_col", "entry_px", "exit_px", "stop_px",
                     "target_px", "stop_dist", "exit_reason", "points", "R_net", "point_value", "commission",
                     "integer_units"]


def generate_candidates(asset: AssetData, logic: TradeLogic, exec_cfg: ExecutionConfig = EXEC) -> pd.DataFrame:
    """
    Evaluate one TradeLogic on every session of an asset.

    Returns one row per *candidate* session: a signal existed at the end of the OR window. `triggered`
    says whether an entry actually happened. Day filters and cross-sectional ranking must run on candidates,
    never on triggered trades only, because triggering is not known at OR time.
    """
    logic.validate()
    spec, p, k = asset.spec, asset.panel, logic.orb_min
    orc = asset.or_stats[k]
    feats = asset.features
    tick = spec.tick_size
    tol = 0.25 * tick                     # comparison tolerance: panel prices are float32
    slip = spec.slippage_ticks * tick

    or_dir = orc["direction"].to_numpy().astype(np.int64)
    if logic.direction_source == "or_candle":
        d = or_dir
    elif logic.direction_source == "gap":
        gd = np.sign(np.nan_to_num(feats["gap"].to_numpy(), nan=0.0)).astype(np.int64)
        d = np.where(gd == or_dir, or_dir, 0)
    else:
        tu = feats["trend_up_prev"].to_numpy()
        td = np.where(np.isnan(tu), 0, np.where(tu > 0.5, 1, -1))
        d = np.where(td == or_dir, or_dir, 0)

    atr = feats["atr_prev"].to_numpy()
    or_range = orc["or_range"].to_numpy()
    raw_stop = logic.stop_param * (atr if logic.stop_type == "atr" else or_range)
    stop_dist = np.ceil(np.round(raw_stop / tick, 6)) * tick
    prev_close = feats["prev_close"].to_numpy()
    cand = ((d != 0) & orc["valid"].to_numpy() & np.isfinite(atr) & (atr > 0)
            & np.isfinite(stop_dist) & (stop_dist >= exec_cfg.min_stop_ticks * tick)
            & np.isfinite(prev_close) & (prev_close >= spec.min_price))
    r = np.flatnonzero(cand)
    if r.size == 0:
        return pd.DataFrame(columns=CANDIDATE_COLUMNS)

    d = d[r].astype(np.float64)
    dd = d[:, None]
    long_ = dd > 0
    trig = _snap(np.where(d > 0, orc["or_high"].to_numpy()[r], orc["or_low"].to_numpy()[r]), tick)
    tg = trig[:, None]
    sd = stop_dist[r]

    # Execution window: columns k … exit_col, re-indexed to j = col − k.
    O, H, L, C = (m[r, k:exec_cfg.exit_col + 1] for m in (p.open, p.high, p.low, p.close))
    m_cols = O.shape[1]
    j = np.arange(m_cols)[None, :]
    last_j = (np.minimum(p.last_col[r], exec_cfg.exit_col) - k)[:, None]
    within = j <= last_j
    rows = np.arange(r.size)

    touch = np.where(long_, H >= tg - tol, L <= tg + tol) & within
    close_beyond = np.where(long_, C > tg + tol, C < tg - tol) & within

    if logic.entry_type == "stop":
        e = _first_true(touch)
        o_e = _snap(_take(O, rows, e), tick)
        entry = np.where(d > 0, np.maximum(trig, o_e), np.minimum(trig, o_e)) + d * slip
        brk = e
    elif logic.entry_type == "close":
        e = _first_true(close_beyond)
        entry = _snap(_take(C, rows, e), tick) + d * slip
        brk = e
    else:
        brk = _first_true(touch)
        retest = (np.where(long_, (L <= tg + tol) & (C > tg + tol), (H >= tg - tol) & (C < tg - tol))
                  & within & (j > brk[:, None]) & (brk[:, None] >= 0))
        e = _first_true(retest)
        entry = _snap(_take(C, rows, e), tick) + d * slip

    ok = (e >= 0) & (e < last_j[:, 0])               # need at least one bar after entry

    if logic.breakout_vol_mult > 0:
        vb = _take(p.volume, r, brk + k)
        va = _take(p.vol_avg_prev, r, brk + k)
        ok &= (brk >= 0) & np.isfinite(va) & (va > 0) & (vb >= logic.breakout_vol_mult * va)
    if logic.use_vwap_filter:
        if logic.entry_type == "stop":            # only the previous bar's VWAP is known at fill time
            vw = _take(p.vwap, r, e + k - 1)
            ref = trig
        else:
            vw = _take(p.vwap, r, e + k)
            ref = _take(C, rows, e)
        ok &= np.isfinite(vw) & np.where(d > 0, ref > vw, ref < vw)

    stop_px = entry - d * sd
    stop_px = np.where(d > 0, np.floor(np.round(stop_px / tick, 6)) * tick, np.ceil(np.round(stop_px / tick, 6)) * tick)
    if logic.target_type == "eod":
        target_px = np.full(r.size, np.nan)
    else:
        dist = logic.target_param * (or_range[r] if logic.target_type == "or_mult" else sd)
        raw_t = entry + d * dist
        target_px = np.where(d > 0, np.ceil(np.round(raw_t / tick, 6)) * tick, np.floor(np.round(raw_t / tick, 6)) * tick)

    e_col = e[:, None]
    stop_from = e_col if logic.entry_type == "stop" else e_col + 1
    stop_hit = np.where(long_, L <= stop_px[:, None] + tol, H >= stop_px[:, None] - tol) & within & (j >= stop_from)
    js = _first_true(stop_hit)
    if logic.target_type == "eod":
        jt = np.full(r.size, -1)
    else:
        tp_ = target_px[:, None]
        tgt_hit = np.where(long_, H >= tp_ + tick - tol, L <= tp_ - tick + tol) & within & (j >= e_col + 1)
        jt = _first_true(tgt_hit)

    big = np.iinfo(np.int64).max
    js_, jt_ = np.where(js >= 0, js, big), np.where(jt >= 0, jt, big)
    is_stop = (js_ < big) & (js_ <= jt_)
    is_tgt = (jt_ < big) & (jt_ < js_)

    o_s = _snap(_take(O, rows, js), tick)
    stop_fill = np.where(js == e, stop_px, np.where(d > 0, np.minimum(stop_px, o_s), np.maximum(stop_px, o_s))) - d * slip
    o_t = _snap(_take(O, rows, jt), tick)
    tgt_fill = np.where(d > 0, np.maximum(target_px, o_t), np.minimum(target_px, o_t))
    lj = last_j[:, 0]
    eod_fill = _snap(_take(C, rows, lj), tick) - d * slip

    exit_px = np.where(is_stop, stop_fill, np.where(is_tgt, tgt_fill, eod_fill))
    exit_j = np.where(is_stop, js, np.where(is_tgt, jt, lj))
    reason = np.where(is_stop, "stop", np.where(is_tgt, "target", "eod"))

    points = d * (exit_px - entry)
    pv, comm = spec.point_value, spec.commission
    r_net = (points * pv - 2 * comm) / (sd * pv)

    nr7 = feats["nr7_prev"].to_numpy()[r]
    out = pd.DataFrame({
        "asset": spec.symbol, "date": p.dates[r], "direction": d.astype(np.int8),
        "or_high": _snap(orc["or_high"].to_numpy()[r], tick), "or_low": _snap(orc["or_low"].to_numpy()[r], tick),
        "or_range": or_range[r], "or_atr": or_range[r] / atr[r], "atr": atr[r],
        "rel_vol": orc["rel_vol"].to_numpy()[r], "nr7_prev": nr7, "triggered": ok,
        "entry_col": np.where(ok, e + k, -1), "exit_col": np.where(ok, exit_j + k, -1),
        "entry_px": np.where(ok, entry, np.nan), "exit_px": np.where(ok, exit_px, np.nan),
        "stop_px": np.where(ok, stop_px, np.nan), "target_px": np.where(ok, target_px, np.nan),
        "stop_dist": sd, "exit_reason": np.where(ok, reason, "none"),
        "points": np.where(ok, points, np.nan), "R_net": np.where(ok, r_net, np.nan),
        "point_value": pv, "commission": comm, "integer_units": spec.integer_units,
    })
    return out


def candidates_for(universe: Iterable[str], logic: TradeLogic) -> pd.DataFrame:
    parts = [c for c in (generate_candidates(assets[s], logic) for s in universe) if not c.empty]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=CANDIDATE_COLUMNS)


def check_invariants(cands: pd.DataFrame, logic: TradeLogic) -> None:
    """Sanity checks on real-data output. They raise AssertionError if a fill rule is violated."""
    t = cands.loc[cands["triggered"]]
    if t.empty:
        return
    d = t["direction"].astype(float)
    trig = np.where(d > 0, t["or_high"], t["or_low"])
    assert (t["entry_col"] >= logic.orb_min).all(), "entry inside the opening range"
    assert (t["exit_col"] >= t["entry_col"]).all(), "exit before entry"
    if logic.entry_type == "stop":
        assert (d * (t["entry_px"] - trig) >= -1e-6).all(), "stop entry filled on the wrong side of the trigger"
    assert (d * (t["entry_px"] - t["stop_px"]) > 0).all(), "stop on the wrong side of entry"
    tgt = t["exit_reason"] == "target"
    assert (d[tgt] * (t.loc[tgt, "exit_px"] - t.loc[tgt, "entry_px"]) > 0).all(), "losing target exit"
    stp = t["exit_reason"] == "stop"
    assert (d[stp] * (t.loc[stp, "exit_px"] - t.loc[stp, "stop_px"]) <= 1e-6).all(), "stop filled better than its level"
    assert np.isfinite(t[["entry_px", "exit_px", "R_net"]].to_numpy()).all(), "non-finite trade values"

## 5. Portfolio simulator & metrics

* **Sizing.** `risk_per_trade × start-of-day equity / (stop distance × point value)`, rounded down to whole
  units. Intraday P&L is not recycled into sizing within the same day.
* **Gross-leverage cap.** Trades are processed in entry-time order. A new position can only use the notional
  left over after positions still open at that minute.
* **Mark-to-market.** Equity is recorded on **every** session in the calendar. A day with no trade is a 0 %
  return, so Sharpe and volatility are not inflated by dropping flat days.
* **Deflated Sharpe Ratio** (Bailey & López de Prado, 2014). The probability that the selected strategy's
  true Sharpe is above zero, after correcting for the number of configurations tried and for non-normal
  returns.

In [7]:
def select_trades(cands: pd.DataFrame, f: DayFilter) -> pd.DataFrame:
    """Apply OR-time filters and the cross-sectional top-N on *candidates*, then keep triggered trades."""
    if cands.empty:
        return cands
    m = (cands["or_atr"] >= f.or_atr_min) & (cands["or_atr"] <= f.or_atr_max)
    if f.require_nr7:
        m &= cands["nr7_prev"] == 1.0
    if f.rel_vol_min > 0:
        m &= cands["rel_vol"] >= f.rel_vol_min
    c = cands.loc[m]
    if f.top_n is not None:
        rank = c.groupby("date")["rel_vol"].rank(ascending=False, method="first", na_option="bottom")
        c = c.loc[rank <= f.top_n]
    return c.loc[c["triggered"]]


@dataclass
class BacktestResult:
    trades: pd.DataFrame
    equity: pd.Series
    returns: pd.Series


def run_portfolio(trades: pd.DataFrame, calendar: pd.DatetimeIndex, pcfg: PortfolioConfig = PORT) -> BacktestResult:
    calendar = pd.DatetimeIndex(calendar).sort_values().unique()
    trades = trades.loc[trades["date"].isin(calendar)].sort_values(["date", "entry_col", "asset"], kind="stable")
    n = len(trades)
    qty = np.zeros(n)
    pnl = np.zeros(n)
    day_pnl = np.zeros(len(calendar))

    if n:
        dpos = calendar.get_indexer(trades["date"])
        entry_col = trades["entry_col"].to_numpy()
        exit_col = trades["exit_col"].to_numpy()
        entry_px = trades["entry_px"].to_numpy()
        points = trades["points"].to_numpy()
        stop_dist = trades["stop_dist"].to_numpy()
        pv = trades["point_value"].to_numpy()
        comm = trades["commission"].to_numpy()
        integer = trades["integer_units"].to_numpy()
        starts = np.flatnonzero(np.r_[True, dpos[1:] != dpos[:-1]])
        ends = np.r_[starts[1:], n]

        equity = pcfg.initial_equity
        for s, e_ in zip(starts, ends):
            dp = dpos[s]
            sod = equity
            if sod <= 0:
                break
            open_exit: list[int] = []
            open_notional: list[float] = []
            for i in range(s, e_):
                in_use = sum(nv for xc, nv in zip(open_exit, open_notional) if xc > entry_col[i])
                unit_notional = entry_px[i] * pv[i]
                q = min(pcfg.risk_per_trade * sod / (stop_dist[i] * pv[i]),
                        max(pcfg.max_gross_leverage * sod - in_use, 0.0) / unit_notional)
                if integer[i]:
                    q = math.floor(q + 1e-9)
                if q <= 0:
                    continue
                qty[i] = q
                pnl[i] = q * (points[i] * pv[i] - 2 * comm[i])
                open_exit.append(exit_col[i])
                open_notional.append(q * unit_notional)
            day_pnl[dp] = pnl[s:e_].sum()
            equity = sod + day_pnl[dp]

    trades = trades.assign(qty=qty, pnl=pnl)
    equity_curve = pd.Series(pcfg.initial_equity + np.cumsum(day_pnl), index=calendar, name="equity")
    prev = equity_curve.shift(1).fillna(pcfg.initial_equity)
    returns = (equity_curve / prev - 1.0).where(prev > 0, 0.0)
    return BacktestResult(trades.loc[trades["qty"] > 0], equity_curve, returns.rename("ret"))


def perf_stats(returns: pd.Series, trades: pd.DataFrame | None = None, periods: int = 252) -> dict:
    """Performance of a daily return series (every session, zeros included) plus trade statistics."""
    rets = returns.dropna()
    out: dict = {"days": int(len(rets))}
    if len(rets) < 2:
        return out
    growth = (1 + rets).cumprod()
    years = len(rets) / periods
    sd = rets.std(ddof=1)
    downside = np.sqrt(np.mean(np.minimum(rets, 0.0) ** 2))
    out.update({
        "total_return_%": 100 * (growth.iloc[-1] - 1),
        "CAGR_%": 100 * (growth.iloc[-1] ** (1 / years) - 1) if growth.iloc[-1] > 0 else np.nan,
        "vol_%": 100 * sd * np.sqrt(periods),
        "sharpe": rets.mean() / sd * np.sqrt(periods) if sd > 0 else np.nan,
        "sortino": rets.mean() / downside * np.sqrt(periods) if downside > 0 else np.nan,
        "MDD_%": 100 * (growth / growth.cummax() - 1).min(),
    })
    if trades is not None:
        t = trades
        out["trades"] = int(len(t))
        out["exposure_%"] = 100 * t["date"].nunique() / len(rets) if len(t) else 0.0
        if len(t):
            gains, losses = t.loc[t["pnl"] > 0, "pnl"].sum(), -t.loc[t["pnl"] < 0, "pnl"].sum()
            out.update({
                "hit_%": 100 * (t["pnl"] > 0).mean(),
                "avg_R": t["R_net"].mean(),
                "R_tstat": (t["R_net"].mean() / (t["R_net"].std(ddof=1) / np.sqrt(len(t)))
                            if len(t) > 1 and t["R_net"].std(ddof=1) > 0 else np.nan),
                "profit_factor": gains / losses if losses > 0 else np.nan,
            })
    return out


def deflated_sharpe(returns: pd.Series, trial_sharpes: np.ndarray, n_trials: int) -> dict:
    """
    Deflated Sharpe Ratio (Bailey & López de Prado, 2014) on per-period (daily) Sharpe ratios.
    trial_sharpes: daily Sharpe of every configuration tried (selection-bias variance estimate).
    """
    r = returns.dropna()
    t = len(r)
    sr = r.mean() / r.std(ddof=1)
    var_sr = np.nanvar(trial_sharpes, ddof=1)
    gamma = 0.5772156649
    n = max(int(n_trials), 2)
    sr0 = np.sqrt(var_sr) * ((1 - gamma) * stats.norm.ppf(1 - 1 / n) + gamma * stats.norm.ppf(1 - 1 / (n * np.e)))
    skew, kurt = stats.skew(r), stats.kurtosis(r, fisher=False)
    denom = np.sqrt(max(1 - skew * sr + (kurt - 1) / 4 * sr ** 2, 1e-12))
    dsr = stats.norm.cdf((sr - sr0) * np.sqrt(t - 1) / denom)
    psr = stats.norm.cdf(sr * np.sqrt(t - 1) / denom)
    return {"daily_SR": sr, "SR0_expected_max": sr0, "PSR(SR>0)": psr, "DSR": dsr, "n_trials": n}


def session_calendar(names: Iterable[str]) -> pd.DatetimeIndex:
    idx = pd.DatetimeIndex([])
    for s in names:
        idx = idx.union(assets[s].panel.dates)
    return idx


def buy_and_hold(asset: AssetData, calendar: pd.DatetimeIndex, initial: float) -> pd.Series:
    """Total-return proxy (files are dividend-adjusted): compounded close-to-close returns."""
    d = asset.daily
    ret = (d["close"] / d["prev_close"] - 1).reindex(calendar).fillna(0.0)
    return (initial * (1 + ret).cumprod()).rename(f"Buy & hold {asset.spec.symbol}")

## 6. Self-tests

Each test builds synthetic sessions with known prices and checks one rule from §4 and §5, plus the
feature-timing contract. Run this cell after any change to the engine.

In [8]:
def _synthetic_asset(day_bars: list[dict[int, tuple[float, float, float, float]]], base: float = 100.0,
                     start: str = "2020-01-01", n_warmup: int = 30, atr: float = 2.0,
                     spec: InstrumentSpec | None = None) -> AssetData:
    """Flat warm-up sessions followed by test sessions whose bars are overridden {col: (o, h, l, c)}."""
    spec = spec or InstrumentSpec("TST", tick_size=0.01, point_value=1.0, commission=0.0, slippage_ticks=1.0)
    dates = pd.bdate_range(start, periods=n_warmup + len(day_bars))
    frames = []
    for i, dt in enumerate(dates):
        ts = dt + pd.to_timedelta(RTH_OPEN_MIN + np.arange(RTH_BARS), unit="min")
        o = np.full(RTH_BARS, base); h = o + 0.05; l = o - 0.05; c = o.copy()
        if i >= n_warmup:
            for col, (bo, bh, bl, bc) in day_bars[i - n_warmup].items():
                o[col], h[col], l[col], c[col] = bo, bh, bl, bc
        frames.append(pd.DataFrame({"open": o, "high": h, "low": l, "close": c, "volume": 1000.0}, index=ts))
    a = build_asset(spec, pd.concat(frames))
    a.features["atr_prev"] = atr                      # pin ATR so stop distances are exact
    return a


def _one(a: AssetData, logic: TradeLogic) -> pd.Series:
    c = generate_candidates(a, logic)
    c = c.loc[c["date"] == a.panel.dates[-1]]
    assert len(c) == 1, f"expected one candidate, got {len(c)}"
    return c.iloc[0]


def _or_up(extra: dict) -> dict:
    """5-minute OR from 100.00 to 101.00 (bullish candle), then flat at 100.50 unless overridden."""
    bars = {0: (100.0, 100.5, 99.9, 100.4), 1: (100.4, 100.8, 100.3, 100.7), 2: (100.7, 101.0, 100.6, 100.9),
            3: (100.9, 100.95, 100.5, 100.6), 4: (100.6, 100.7, 100.5, 100.65)}
    for col in range(5, RTH_BARS):
        bars[col] = (100.5, 100.55, 100.45, 100.5)
    bars.update(extra)
    return bars


def run_self_tests() -> None:
    lg = TradeLogic(orb_min=5, entry_type="stop", stop_type="atr", stop_param=0.5, target_type="r_mult", target_param=1.0)

    # 1. Stop entry, gap through the trigger → fill at the bar open + 1 tick.
    a = _synthetic_asset([_or_up({10: (101.20, 101.30, 101.10, 101.25)})])
    t = _one(a, lg)
    assert t.triggered and t.entry_col == 10 and abs(t.entry_px - 101.21) < 1e-9, t

    # 2. Same bar trades through both stop and target → stop wins.
    a = _synthetic_asset([_or_up({10: (100.95, 101.05, 100.90, 101.02), 11: (101.0, 103.0, 99.0, 101.0)})])
    t = _one(a, lg)
    assert t.exit_reason == "stop" and t.exit_col == 11, t

    # 3. Target touched exactly but not traded through → no fill. One tick through → filled at the target.
    entry = 101.01; target = entry + 1.0
    a = _synthetic_asset([_or_up({10: (100.95, 101.05, 100.90, 101.02), 12: (101.5, target, 101.4, 101.9)})])
    assert _one(a, lg).exit_reason == "eod"
    a = _synthetic_asset([_or_up({10: (100.95, 101.05, 100.90, 101.02), 12: (101.5, target + 0.01, 101.4, 101.9)})])
    t = _one(a, lg)
    assert t.exit_reason == "target" and abs(t.exit_px - target) < 1e-9, t

    # 4. Stop entry whose own bar trades through the stop → stopped on the entry bar.
    a = _synthetic_asset([_or_up({10: (100.95, 101.10, 99.50, 100.0)})])
    t = _one(a, lg)
    assert t.exit_reason == "stop" and t.exit_col == 10 and abs(t.exit_px - (entry - 1.0 - 0.01)) < 1e-9, t

    # 5. EOD exit: no stop or target → last bar close minus slippage.
    lg_eod = replace(lg, target_type="eod", target_param=0.0, stop_param=5.0)
    a = _synthetic_asset([_or_up({10: (100.95, 101.05, 100.90, 101.02), 389: (101.5, 101.6, 101.4, 101.55)})])
    t = _one(a, lg_eod)
    assert t.exit_reason == "eod" and t.exit_col == 389 and abs(t.exit_px - 101.54) < 1e-9, t

    # 6. Close entry needs a close beyond the level. A wick-only touch does not trigger.
    lg_close = replace(lg, entry_type="close")
    a = _synthetic_asset([_or_up({10: (100.95, 101.20, 100.90, 100.98), 11: (100.98, 101.10, 100.95, 101.05)})])
    t = _one(a, lg_close)
    assert t.entry_col == 11 and abs(t.entry_px - 101.06) < 1e-9, t

    # 7. No look-ahead: corrupting the last session must not change any earlier feature or OR stat.
    a1 = _synthetic_asset([_or_up({}), _or_up({})], n_warmup=80)
    a2 = _synthetic_asset([_or_up({}), _or_up({k: (150.0, 160.0, 140.0, 155.0) for k in range(RTH_BARS)})], n_warmup=80)
    f1 = build_daily_features(a1.daily).iloc[:-1]
    f2 = build_daily_features(a2.daily).iloc[:-1]
    pd.testing.assert_frame_equal(f1, f2)
    g1 = build_daily_features(a1.daily).iloc[-1]
    g2 = build_daily_features(a2.daily).iloc[-1]
    prior_only = [c for c in g1.index if c not in ("gap", "gap_atr", "gap_pct")]
    pd.testing.assert_series_equal(g1[prior_only], g2[prior_only], check_names=False)
    for k in OR_WINDOWS:
        pd.testing.assert_frame_equal(a1.or_stats[k].iloc[:-1], a2.or_stats[k].iloc[:-1])

    # 8. top_n ranks candidates *before* triggering is known.
    cands = pd.DataFrame({"date": pd.Timestamp("2020-01-01"), "asset": ["A", "B"], "or_atr": 0.5,
                          "nr7_prev": 0.0, "rel_vol": [3.0, 1.0], "triggered": [False, True]})
    assert select_trades(cands, DayFilter(top_n=1)).empty, "top-N must not skip to a triggered lower-ranked asset"

    # 9. Gross leverage cap and integer sizing.
    tr = pd.DataFrame({"asset": ["A", "B"], "date": pd.Timestamp("2020-01-02"), "entry_col": [10, 20],
                       "exit_col": [100, 100], "entry_px": [100.0, 100.0], "points": [1.0, 1.0],
                       "stop_dist": [0.01, 0.01], "point_value": 1.0, "commission": 0.0,
                       "integer_units": True, "R_net": 100.0})
    res = run_portfolio(tr, pd.DatetimeIndex(["2020-01-02"]), PortfolioConfig(10_000, 0.01, 1.0))
    assert res.trades["qty"].tolist() == [100], res.trades     # second trade gets no capacity
    assert abs(res.equity.iloc[-1] - 10_100) < 1e-9

    print("All self-tests passed ✓")


run_self_tests()

All self-tests passed ✓


## 7. Search protocol

1. Fix a chronological split **before** looking at any results: the first 60 % of sessions are in-sample
   (IS), the rest out-of-sample (OOS).
2. Sample configurations from the grid. Each configuration is simulated once over the full period
   (features are causal, so OOS days never inform IS days), and its metrics are split at the boundary.
3. **Select on IS only**: the highest IS Sharpe among configurations with at least `MIN_IS_TRADES` trades.
4. Report that one configuration's OOS result, its Deflated Sharpe on IS, and how well IS rank predicts
   OOS rank across all configurations (Spearman ρ). If ρ is near zero, IS optimisation is mostly fitting noise.

In [9]:
LOGIC_GRID = {
    "orb_min": list(OR_WINDOWS),
    "direction_source": ["or_candle", "gap", "trend"],
    "entry_type": ["stop", "close", "retest"],
    "breakout_vol_mult": [0.0, 1.2, 1.5],
    "use_vwap_filter": [False, True],
    "stop_type": ["atr"],
    "stop_param": [0.10, 0.25, 0.50],
    "target": [("eod", 0.0), ("or_mult", 0.5), ("or_mult", 1.0), ("or_mult", 1.5), ("or_mult", 3.0),
               ("r_mult", 1.0), ("r_mult", 1.5), ("r_mult", 2.0), ("r_mult", 3.0)],
}
FILTER_GRID = {
    "or_atr": [(0.0, math.inf), (0.3, 1.5), (0.5, 1.2), (0.0, 0.6)],
    "require_nr7": [False, True],
    "rel_vol_min": [0.0, 1.0, 1.5, 2.0],
    "top_n": [3, 5, 10],
}
MIN_IS_TRADES = 100
IS_FRACTION = 0.60


def enumerate_logics() -> list[TradeLogic]:
    out = []
    keys = list(LOGIC_GRID)
    for combo in itertools.product(*LOGIC_GRID.values()):
        d = dict(zip(keys, combo))
        tt, tp = d.pop("target")
        lg = TradeLogic(**d, target_type=tt, target_param=tp)
        try:
            lg.validate()
        except ValueError:
            continue
        out.append(lg)
    return out


def enumerate_filters(single_asset: bool) -> list[DayFilter]:
    out = []
    for (lo, hi), nr7, rv, top in itertools.product(*FILTER_GRID.values()):
        out.append(DayFilter(lo, hi, nr7, rv, None if single_asset else top))
    return list(dict.fromkeys(out))                  # top_n collapses for one asset


def split_date_for(calendar: pd.DatetimeIndex, frac: float = IS_FRACTION) -> pd.Timestamp:
    return calendar[int(len(calendar) * frac)]


def run_search(universe: list[str], n_logic: int, filters_per_logic: int, seed: int = SEED,
               pcfg: PortfolioConfig = PORT) -> tuple[pd.DataFrame, pd.Timestamp]:
    rng = np.random.default_rng(seed)
    logics = enumerate_logics()
    filters = enumerate_filters(single_asset=len(universe) == 1)
    chosen = rng.choice(len(logics), size=min(n_logic, len(logics)), replace=False)
    calendar = session_calendar(universe)
    split = split_date_for(calendar)
    is_mask = calendar < split
    log.info("Search: %d logics × %d filters | %d logic / %d filter combos available | IS < %s ≤ OOS",
             len(chosen), filters_per_logic, len(logics), len(filters), split.date())

    rows: list[dict] = []
    t0 = time.perf_counter()
    for n_done, li in enumerate(chosen, 1):
        logic = logics[li]
        cands = candidates_for(universe, logic)
        for fi in rng.choice(len(filters), size=min(filters_per_logic, len(filters)), replace=False):
            filt = filters[fi]
            trades = select_trades(cands, filt)
            res = run_portfolio(trades, calendar, pcfg)
            t_is = res.trades.loc[res.trades["date"] < split]
            t_oos = res.trades.loc[res.trades["date"] >= split]
            s_is = perf_stats(res.returns[is_mask], t_is)
            s_oos = perf_stats(res.returns[~is_mask], t_oos)
            r_is = res.returns[is_mask]
            rows.append({"logic_id": int(li), **asdict(logic), **asdict(filt),
                         "is_daily_sr": r_is.mean() / r_is.std() if r_is.std() > 0 else np.nan,
                         **{f"is_{k}": v for k, v in s_is.items()},
                         **{f"oos_{k}": v for k, v in s_oos.items()}})
        if n_done % max(1, len(chosen) // 10) == 0:
            log.info("  %d/%d logics  (%.0fs)", n_done, len(chosen), time.perf_counter() - t0)
    return pd.DataFrame(rows), split


def config_from_row(row: pd.Series) -> tuple[TradeLogic, DayFilter]:
    lg = TradeLogic(**{f.name: (row[f.name].item() if hasattr(row[f.name], "item") else row[f.name])
                       for f in TradeLogic.__dataclass_fields__.values()})
    top = row["top_n"]
    flt = DayFilter(float(row["or_atr_min"]), float(row["or_atr_max"]), bool(row["require_nr7"]),
                    float(row["rel_vol_min"]), None if pd.isna(top) else int(top))
    return lg, flt


def eligible(results: pd.DataFrame) -> pd.DataFrame:
    elig = results.loc[(results["is_trades"] >= MIN_IS_TRADES) & results["is_sharpe"].notna()]
    if elig.empty:
        raise RuntimeError(f"No configuration has ≥ {MIN_IS_TRADES} IS trades. Lower MIN_IS_TRADES or widen the grid.")
    return elig


def select_config(results: pd.DataFrame) -> tuple[TradeLogic, DayFilter]:
    """The single selection rule: highest IS Sharpe among eligible configurations. Uses no OOS columns."""
    elig = eligible(results)
    return config_from_row(elig.loc[elig["is_sharpe"].idxmax()])


def evaluate_search(results: pd.DataFrame, split: pd.Timestamp, universe: list[str], label: str,
                    benchmark: str = BENCHMARK) -> tuple[BacktestResult, pd.Series]:
    elig = eligible(results)
    logic, filt = select_config(results)

    rho, pval = stats.spearmanr(elig["is_sharpe"], elig["oos_sharpe"], nan_policy="omit")
    top_decile = elig.nlargest(max(1, len(elig) // 10), "is_sharpe")
    print(f"{label}: {len(results):,} configurations evaluated, {len(elig):,} eligible (≥ {MIN_IS_TRADES} IS trades)")
    print(f"IS→OOS Sharpe rank correlation (Spearman ρ): {rho:+.3f}  (p = {pval:.3g})")
    print(f"Median OOS Sharpe — all eligible: {elig['oos_sharpe'].median():+.3f} | "
          f"top IS decile: {top_decile['oos_sharpe'].median():+.3f}")
    print(f"Share of eligible configs with OOS Sharpe > 0: {100 * (elig['oos_sharpe'] > 0).mean():.1f} %\n")

    print("Selected configuration (by IS Sharpe only):")
    print("  ", logic)
    print("  ", filt)

    calendar = session_calendar(universe)
    cands = candidates_for(universe, logic)
    check_invariants(cands, logic)
    res = run_portfolio(select_trades(cands, filt), calendar)
    is_mask = calendar < split
    bh = buy_and_hold(assets[benchmark], calendar, PORT.initial_equity)
    bh_ret = bh.pct_change().fillna(bh.iloc[0] / PORT.initial_equity - 1)
    table = pd.DataFrame({
        "Strategy — IS": perf_stats(res.returns[is_mask], res.trades[res.trades["date"] < split]),
        "Strategy — OOS": perf_stats(res.returns[~is_mask], res.trades[res.trades["date"] >= split]),
        f"B&H {benchmark} — IS": perf_stats(bh_ret[is_mask]),
        f"B&H {benchmark} — OOS": perf_stats(bh_ret[~is_mask]),
    }).T
    display(table)

    dsr = deflated_sharpe(res.returns[is_mask], results["is_daily_sr"].to_numpy(), len(results))
    print("Deflated Sharpe on IS (selection-bias adjusted):")
    display(pd.Series(dsr).to_frame("value"))
    return res, bh


def plot_equity(curves: dict[str, pd.Series], split: pd.Timestamp | None, title: str) -> None:
    fig = go.Figure()
    for name, s in curves.items():
        fig.add_trace(go.Scatter(x=s.index, y=s.values, name=name, mode="lines"))
    if split is not None:
        fig.add_vrect(x0=split, x1=max(s.index.max() for s in curves.values()), fillcolor="grey",
                      opacity=0.08, line_width=0, annotation_text="out-of-sample", annotation_position="top left")
    fig.update_layout(title=title, yaxis_type="log", yaxis_title="Equity ($, log scale)", template="plotly_white",
                      height=560, hovermode="x unified",
                      legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0))
    fig.update_yaxes(tickformat="$,.0f")
    fig.show()


def trade_breakdown(trades: pd.DataFrame, split: pd.Timestamp) -> None:
    t = trades.assign(period=np.where(trades["date"] < split, "IS", "OOS"), year=trades["date"].dt.year)
    print("Exit reasons (R after costs):")
    display(t.groupby(["period", "exit_reason"])["R_net"].agg(["count", "mean", "sum"]))
    if t["asset"].nunique() > 1:
        print("Per asset:")
        display(t.groupby(["asset", "period"]).agg(trades=("pnl", "size"), pnl=("pnl", "sum"),
                                                    avg_R=("R_net", "mean"),
                                                    hit=("pnl", lambda s: (s > 0).mean())).unstack("period"))
    print("Per year:")
    display(t.groupby("year").agg(trades=("pnl", "size"), pnl=("pnl", "sum"), avg_R=("R_net", "mean"),
                                  hit=("pnl", lambda s: (s > 0).mean())))

## 8. Experiment A — ETF universe

Universe: the ten ETFs. Trades are ranked across assets each morning by OR relative volume (top-N).
Benchmark: SPY buy & hold.

> Caveat: from 2022-03-07 the ETF bars are IEX-only. Highs and lows printed on one venue are an approximation
> of the consolidated tape, and sessions with sparse bars are excluded by the coverage gate (see the
> data-quality table above).

In [10]:
ETF_UNIVERSE = [s for s in ETF_SYMBOLS if s in assets]
N_LOGIC, FILTERS_PER_LOGIC = (40, 3) if QUICK_RUN else (600, 4)

t0 = time.perf_counter()
etf_results, etf_split = run_search(ETF_UNIVERSE, N_LOGIC, FILTERS_PER_LOGIC)
log.info("Experiment A search finished in %.0fs", time.perf_counter() - t0)

22:25:39  INFO    Search: 600 logics × 4 filters | 5670 logic / 96 filter combos available | IS < 2017-03-22 ≤ OOS


22:25:49  INFO      60/600 logics  (10s)


22:25:58  INFO      120/600 logics  (18s)


22:26:06  INFO      180/600 logics  (27s)


22:26:16  INFO      240/600 logics  (37s)


22:26:26  INFO      300/600 logics  (46s)


... (2 outputs left out)

22:26:55  INFO      480/600 logics  (75s)


22:27:08  INFO      540/600 logics  (89s)


22:27:18  INFO      600/600 logics  (99s)


22:27:18  INFO    Experiment A search finished in 99s


In [11]:
etf_best, spy_bh = evaluate_search(etf_results, etf_split, ETF_UNIVERSE, "Experiment A (ETFs)")
plot_equity({"Selected ORB config": etf_best.equity, spy_bh.name: spy_bh}, etf_split,
            "Experiment A — ETF ORB (selected on IS only) vs SPY buy & hold")
trade_breakdown(etf_best.trades, etf_split)

print("Top 10 configurations by IS Sharpe, with their OOS outcome:")
show = ["orb_min", "direction_source", "entry_type", "breakout_vol_mult", "use_vwap_filter", "stop_param",
        "target_type", "target_param", "or_atr_min", "or_atr_max", "require_nr7", "rel_vol_min", "top_n",
        "is_trades", "is_sharpe", "oos_trades", "oos_sharpe", "oos_CAGR_%", "oos_MDD_%"]
display(etf_results.loc[etf_results["is_trades"] >= MIN_IS_TRADES].nlargest(10, "is_sharpe")[show])

Experiment A (ETFs): 2,400 configurations evaluated, 1,602 eligible (≥ 100 IS trades)
IS→OOS Sharpe rank correlation (Spearman ρ): +0.480  (p = 8.64e-93)
Median OOS Sharpe — all eligible: +0.002 | top IS decile: +0.220
Share of eligible configs with OOS Sharpe > 0: 50.0 %

Selected configuration (by IS Sharpe only):
   TradeLogic(orb_min=60, direction_source='gap', entry_type='close', breakout_vol_mult=1.5, use_vwap_filter=True, stop_type='atr', stop_param=0.5, target_type='eod', target_param=0.0)
   DayFilter(or_atr_min=0.0, or_atr_max=0.6, require_nr7=False, rel_vol_min=0.0, top_n=10)


,days,total_return_%,CAGR_%,vol_%,sharpe,sortino,MDD_%,trades,exposure_%,hit_%,avg_R,R_tstat,profit_factor
Strategy — IS,"3,581.0000",397.1499,11.9471,14.4747,0.8515,1.4598,-32.2047,"3,425.0000",54.5378,50.7445,0.0467,3.4460,1.1089
Strategy — OOS,"2,388.0000",0.8398,0.0883,10.6912,0.0614,0.0956,-30.5818,"1,788.0000",50.5444,48.6577,-0.0108,-0.6129,1.0021
B&H SPY — IS,"3,581.0000",250.7810,9.2333,18.5596,0.5689,0.7988,-55.2046,NaN,NaN,NaN,NaN,NaN,NaN
B&H SPY — OOS,"2,388.0000",258.9058,14.4368,18.4254,0.8246,1.1471,-33.8463,NaN,NaN,NaN,NaN,NaN,NaN


Deflated Sharpe on IS (selection-bias adjusted):


,value
daily_SR,0.0536
SR0_expected_max,0.1490
PSR(SR>0),0.9996
DSR,0.0000
n_trials,"2,400.0000"


Exit reasons (R after costs):


count    mean       sum
period exit_reason                         
IS     eod           2859  0.2641  754.9515
       stop           566 -1.0511 -594.9453
OOS    eod           1499  0.1896  284.2384
       stop           289 -1.0504 -303.5564

... (2 outputs left out)

Per year:


,trades,pnl,avg_R,hit
year,,,,
2003,97,"6,569.6080",0.2590,0.6804
2004,150,"7,293.3450",0.1438,0.5867
2005,183,"1,728.3750",0.0330,0.4809
2006,211,"8,122.3320",0.0856,0.4834
2007,228,"15,027.7940",0.1274,0.5307
2008,289,"47,590.3130",0.1960,0.5709
2009,261,"9,547.9180",0.0185,0.5096
2010,271,"-2,787.3330",-0.0024,0.5018
2011,321,"25,251.3190",0.0825,0.5327


Top 10 configurations by IS Sharpe, with their OOS outcome:


,orb_min,direction_source,entry_type,breakout_vol_mult,use_vwap_filter,stop_param,target_type,target_param,or_atr_min,or_atr_max,require_nr7,rel_vol_min,top_n,is_trades,is_sharpe,oos_trades,oos_sharpe,oos_CAGR_%,oos_MDD_%
935,60,gap,close,1.5000,True,0.5000,eod,0.0000,0.0000,0.6000,False,0.0000,10,3425,0.8515,1788,0.0614,0.0883,-30.5818
933,60,gap,close,1.5000,True,0.5000,eod,0.0000,0.3000,1.5000,False,1.0000,5,1762,0.8341,840,-0.2144,-2.0242,-26.9568
1189,15,gap,close,1.2000,True,0.5000,r_mult,3.0000,0.0000,inf,False,1.5000,10,653,0.8230,333,0.3350,1.9368,-14.4279
1729,60,gap,close,1.5000,True,0.2500,or_mult,1.5000,0.5000,1.2000,False,0.0000,5,1339,0.7156,704,-0.1335,-1.9612,-28.0672
1289,60,gap,retest,1.2000,True,0.2500,r_mult,3.0000,0.0000,0.6000,False,0.0000,3,2739,0.7038,1598,0.1291,0.8371,-30.9612
1004,15,gap,close,1.2000,False,0.5000,eod,0.0000,0.0000,inf,False,2.0000,3,242,0.6944,144,0.1101,0.3498,-11.5069
2340,15,gap,retest,0.0000,False,0.5000,r_mult,2.0000,0.0000,0.6000,False,1.0000,3,2940,0.6937,1897,0.2308,2.2518,-35.8821
270,60,gap,retest,1.5000,False,0.5000,or_mult,3.0000,0.0000,0.6000,False,0.0000,3,2531,0.6781,1376,-0.3685,-3.6057,-31.0256
1643,45,trend,close,1.5000,True,0.2500,or_mult,1.0000,0.0000,0.6000,False,1.5000,10,272,0.6503,163,0.5370,2.3206,-6.1910
934,60,gap,close,1.5000,True,0.5000,eod,0.0000,0.5000,1.2000,False,1.0000,5,1043,0.6499,517,-0.3923,-2.6748,-23.5789


## 9. Experiment B — Dynamic timeframe selection (the DynamicORB hypothesis)

**Question.** Using only information available at 09:30, can a model choose which OR window
(5/15/30/45/60 min) to trade on each ETF today, or skip it, better than simple rules?

**Design**
* **Base trade logic = Experiment A's selection** (chosen on A's in-sample period only). Only `orb_min`
  varies, and A's day filters are dropped because the model decides which rows to trade. A fixed a-priori
  base (Zarattini-style 0.10 × ATR stop, exit at the close) was evaluated first. It loses about 0.15 R per
  trade after ETF costs in every window, so choosing among its windows says nothing about the hypothesis.
* **Results are reported only for sessions after A's IS/OOS split.** The base logic was selected on earlier
  data, so the evaluation rows never influenced any choice. Models train on all earlier years, including A's
  in-sample period.
* One row per (session, ETF). For each OR window the target is the after-cost `R_net` of that window's trade,
  and **0 if it did not trigger** (a committed choice that does not fire earns nothing).
* **Model.** One Random Forest *regressor* per window estimates expected R from the causal daily features plus
  an ETF indicator, pooled across ETFs for sample size. The policy trades the window with the highest
  prediction, and a variant skips when every prediction is ≤ 0. Predicting expected value directly avoids the
  failure mode of an "argmax window" classifier, which collapses to the majority "skip" class.
* **Walk-forward**: an expanding window retrained each calendar year. Each test year is predicted by models
  that have only seen earlier years.
* Baselines over the same rows: each static window, the **trailing-best** window per ETF (highest mean R in
  the training data, re-chosen yearly), and the oracle (upper bound, not achievable).
* Expanding-window training starts from 2003, so every test year has at least ~13 years of history.
* Significance tests run on **daily** totals (summed over ETFs). Rows from the same day are not independent.

In [12]:
ML_BASE, _ = select_config(etf_results)            # IS-only selection from Experiment A
print(f"Base trade logic (from Experiment A, IS-selected): {ML_BASE}")
MIN_ML_SESSIONS = 1_000                   # ETFs with fewer tradable sessions are too sparse to model (CPER)
ML_UNIVERSE = [s for s in ETF_UNIVERSE if assets[s].session_ok.sum() >= MIN_ML_SESSIONS]
ML_TEST_START = etf_split                           # evaluate strictly after A's IS period
R_CLIP = (-2.0, 6.0)                      # limit the influence of rare outsized R days on the fit
RF_PARAMS = dict(n_estimators=60 if QUICK_RUN else 200, max_depth=5, min_samples_leaf=100,
                 max_features="sqrt", random_state=SEED, n_jobs=-1)
print(f"ML universe ({len(ML_UNIVERSE)}): {ML_UNIVERSE}")

cand_by_tf = {k: candidates_for(ML_UNIVERSE, replace(ML_BASE, orb_min=k)) for k in OR_WINDOWS}
for k, c in cand_by_tf.items():
    check_invariants(c, replace(ML_BASE, orb_min=k))

feature_frames = []
for s in ML_UNIVERSE:
    a = assets[s]
    f = a.features.loc[a.panel.dates[a.session_ok], FEATURE_COLUMNS].dropna()
    feature_frames.append(f.assign(asset=s).set_index("asset", append=True))
X_base = pd.concat(feature_frames).sort_index()                       # index: (date, asset)
asset_dummies = pd.get_dummies(pd.Series(X_base.index.get_level_values("asset"), index=X_base.index),
                               prefix="etf", dtype=float)
X_all = pd.concat([X_base, asset_dummies], axis=1)
ML_FEATURES = list(X_all.columns)

R = pd.DataFrame({k: c.loc[c["triggered"]].set_index(["date", "asset"])["R_net"].reindex(X_all.index).fillna(0.0)
                  for k, c in cand_by_tf.items()})
row_dates = X_all.index.get_level_values("date")
print(f"Rows (session × ETF) with complete features: {len(X_all):,}  "
      f"({row_dates.min().date()} → {row_dates.max().date()})")
print("Mean R_net per row by window over the whole sample (hindsight view, for orientation only):")
display(R.agg(["mean", "std"]).T.assign(trigger_rate=(R != 0).mean()).T)

Base trade logic (from Experiment A, IS-selected): TradeLogic(orb_min=60, direction_source='gap', entry_type='close', breakout_vol_mult=1.5, use_vwap_filter=True, stop_type='atr', stop_param=0.5, target_type='eod', target_param=0.0)
ML universe (9): ['DIA', 'GLD', 'IEF', 'IWM', 'QQQ', 'SHY', 'SLV', 'SPY', 'TLT']


Rows (session × ETF) with complete features: 38,776  (2003-04-02 → 2026-09-21)
Mean R_net per row by window over the whole sample (hindsight view, for orientation only):


,5,15,30,45,60
mean,0.0030,0.0018,0.0048,0.0047,0.0047
std,0.3132,0.3249,0.3853,0.3496,0.3469
trigger_rate,0.1058,0.1300,0.1865,0.1762,0.1813


In [13]:
test_mask = row_dates >= ML_TEST_START
test_idx = X_all.index[test_mask]
test_years = sorted(row_dates[test_mask].year.unique())
pred_R = pd.DataFrame(np.nan, index=test_idx, columns=list(OR_WINDOWS))
trailing_choice = pd.Series(0, index=test_idx, dtype=int)
importances = []
t0 = time.perf_counter()
for yr in test_years:
    tr_mask = row_dates < pd.Timestamp(year=yr, month=1, day=1)
    te_mask = test_mask & (row_dates.year == yr)
    te_idx = X_all.index[te_mask]
    for k in OR_WINDOWS:
        model = RandomForestRegressor(**RF_PARAMS).fit(X_all.loc[tr_mask], R.loc[tr_mask, k].clip(*R_CLIP))
        pred_R.loc[te_idx, k] = model.predict(X_all.loc[te_mask])
        importances.append(pd.Series(model.feature_importances_, index=ML_FEATURES))
    train_means = R.loc[tr_mask].groupby(level="asset").mean()
    per_asset_best = train_means.idxmax(axis=1)
    pooled_best = int(R.loc[tr_mask].mean().idxmax())
    trailing_choice.loc[te_idx] = (per_asset_best.reindex(te_idx.get_level_values("asset"))
                                   .fillna(pooled_best).astype(int).to_numpy())
log.info("Walk-forward fits: %d years × %d windows in %.0fs", len(test_years), len(OR_WINDOWS),
         time.perf_counter() - t0)

R_test = R.loc[test_idx]
ml_choice = pred_R.idxmax(axis=1).astype(int)
ml_choice_skip = ml_choice.where(pred_R.max(axis=1) > 0, 0)


def realised(choice: pd.Series) -> pd.Series:
    """R earned per row by committing to `choice` (0 = skip)."""
    vals = np.zeros(len(choice))
    for k in OR_WINDOWS:
        m = (choice == k).to_numpy()
        vals[m] = R_test.loc[m, k].to_numpy()
    return pd.Series(vals, index=choice.index)


policies = {"ML: best predicted window": realised(ml_choice),
            "ML: best predicted window, skip if ≤ 0": realised(ml_choice_skip),
            "Trailing-best window (per ETF)": realised(trailing_choice),
            **{(f"Static {k}m (Experiment A's window)" if k == ML_BASE.orb_min else f"Static {k}m"): R_test[k]
               for k in OR_WINDOWS},
            "Oracle (upper bound)": R_test.max(axis=1).clip(lower=0)}


def r_stats(r: pd.Series) -> dict:
    traded = r[r != 0]
    daily = r.groupby(level="date").sum()
    sd = daily.std()
    return {"mean_R/row": r.mean(), "total_R": r.sum(), "rows_traded_%": 100 * len(traded) / len(r),
            "hit_% (traded)": 100 * (traded > 0).mean() if len(traded) else np.nan,
            "daily_SR_ann": daily.mean() / sd * np.sqrt(252) if sd > 0 else np.nan,
            "t_stat (daily)": daily.mean() / (sd / np.sqrt(len(daily))) if sd > 0 else np.nan}


test_dates = test_idx.get_level_values("date")
print(f"Walk-forward test period: {test_dates.min().date()} → {test_dates.max().date()} "
      f"({test_dates.nunique():,} sessions, {len(test_idx):,} rows)")
display(pd.DataFrame({k: r_stats(v) for k, v in policies.items()}).T)

for ml_name in ("ML: best predicted window", "ML: best predicted window, skip if ≤ 0"):
    for base in ("Trailing-best window (per ETF)", f"Static {ML_BASE.orb_min}m (Experiment A's window)"):
        diff = (policies[ml_name] - policies[base]).groupby(level="date").sum()
        t, p = stats.ttest_1samp(diff, 0.0)
        print(f"Paired daily test, {ml_name} − {base}: mean ΔR/day = {diff.mean():+.4f}, t = {t:+.2f}, p = {p:.3f}")

# Does the model rank outcomes at all? Realised R by predicted-R quintile (all windows pooled).
pooled = pd.DataFrame({"pred": pred_R.to_numpy().ravel(), "real": R_test.to_numpy().ravel()})
pooled["quintile"] = pd.qcut(pooled["pred"].rank(method="first"), 5, labels=[f"Q{i}" for i in range(1, 6)])
rho, p_rho = stats.spearmanr(pooled["pred"], pooled["real"])
print(f"\nSignal check: Spearman(pred, realised) = {rho:+.4f} (p = {p_rho:.3g}). The p-value is optimistic, "
      "because the five windows of one row, and the ETFs of one day, are correlated.")
print("Realised mean R by predicted-R quintile (Q5 = highest predictions):")
display(pooled.groupby("quintile", observed=True)["real"].agg(["count", "mean"]).T)
print("Window chosen by the skip policy (share of rows; 0 = skip):")
display((ml_choice_skip.value_counts(normalize=True).sort_index() * 100).rename("% of rows").to_frame().T)
print("Mean impurity importance across all fits (biased toward continuous features; indicative only):")
display(pd.concat(importances, axis=1).mean(axis=1).sort_values(ascending=False).to_frame("importance").T)

22:28:22  INFO    Walk-forward fits: 10 years × 5 windows in 63s


Walk-forward test period: 2017-03-22 → 2026-09-21 (2,386 sessions, 15,754 rows)


,mean_R/row,total_R,rows_traded_%,hit_% (traded),daily_SR_ann,t_stat (daily)
ML: best predicted window,-0.0001,-2.1913,14.3519,48.6953,-0.0151,-0.0465
"ML: best predicted window, skip if ≤ 0",0.0018,27.9886,12.8666,49.7780,0.2051,0.6310
Trailing-best window (per ETF),0.0015,23.7685,14.4090,49.2070,0.1611,0.4956
Static 5m,0.0012,19.6071,7.2236,47.4517,0.1854,0.5706
Static 15m,-0.0004,-6.4392,10.4164,47.2882,-0.0517,-0.1591
Static 30m,-0.0007,-11.5419,15.8753,47.6210,-0.0697,-0.2145
Static 45m,-0.0021,-33.4796,15.1708,48.4937,-0.2201,-0.6774
Static 60m (Experiment A's window),-0.0024,-37.8732,15.5580,47.7356,-0.2457,-0.7560
Oracle (upper bound),0.1164,"1,834.3211",16.5736,100.0000,9.2253,28.3869


Paired daily test, ML: best predicted window − Trailing-best window (per ETF): mean ΔR/day = -0.0109, t = -0.72, p = 0.473
Paired daily test, ML: best predicted window − Static 60m (Experiment A's window): mean ΔR/day = +0.0150, t = +0.97, p = 0.334
Paired daily test, ML: best predicted window, skip if ≤ 0 − Trailing-best window (per ETF): mean ΔR/day = +0.0018, t = +0.12, p = 0.907
Paired daily test, ML: best predicted window, skip if ≤ 0 − Static 60m (Experiment A's window): mean ΔR/day = +0.0276, t = +1.80, p = 0.071

Signal check: Spearman(pred, realised) = +0.0289 (p = 5.22e-16). The p-value is optimistic, because the five windows of one row, and the ETFs of one day, are correlated.
Realised mean R by predicted-R quintile (Q5 = highest predictions):


quintile,Q1,Q2,Q3,Q4,Q5
count,"15,754.0000","15,754.0000","15,754.0000","15,754.0000","15,754.0000"
mean,-0.0154,-0.0027,-0.0002,0.0075,0.0064


Window chosen by the skip policy (share of rows; 0 = skip):


,0,5,15,30,45,60
% of rows,15.4373,7.7504,7.0077,17.3353,19.1126,33.3566


Mean impurity importance across all fits (biased toward continuous features; indicative only):


,gap_atr,rv20_prev,gap_pct,trend_z_prev,atr_ratio_prev,hurst_prev,prior_range_atr,prior_close_loc,rv_ratio_prev,etf_SHY,etf_SLV,dow,etf_TLT,etf_SPY,etf_DIA,nr7_prev,etf_GLD,etf_QQQ,etf_IWM,etf_IEF
importance,0.1276,0.1255,0.1156,0.1143,0.0974,0.0832,0.0764,0.0736,0.0660,0.0288,0.0288,0.0277,0.0081,0.0068,0.0054,0.0037,0.0034,0.0030,0.0028,0.0019


In [14]:
def policy_trades(choice: pd.Series) -> pd.DataFrame:
    parts = []
    for k in OR_WINDOWS:
        c = cand_by_tf[k]
        keys = choice.index[choice == k]
        sel = c.loc[c["triggered"] & pd.MultiIndex.from_frame(c[["date", "asset"]]).isin(keys)]
        if not sel.empty:
            parts.append(sel)
    return pd.concat(parts, ignore_index=True) if parts else cand_by_tf[5].iloc[:0]


cal_test = pd.DatetimeIndex(test_dates.unique()).sort_values()
backtests = {
    "ML: best predicted window, skip if ≤ 0": run_portfolio(policy_trades(ml_choice_skip), cal_test),
    "ML: best predicted window": run_portfolio(policy_trades(ml_choice), cal_test),
    "Trailing-best window (per ETF)": run_portfolio(policy_trades(trailing_choice), cal_test),
    f"Static {ML_BASE.orb_min}m (Experiment A's window)":
        run_portfolio(policy_trades(pd.Series(ML_BASE.orb_min, index=test_idx)), cal_test),
}
spy_bh_test = buy_and_hold(assets[BENCHMARK], cal_test, PORT.initial_equity)
summary_b = pd.DataFrame({k: perf_stats(b.returns, b.trades) for k, b in backtests.items()})
summary_b[spy_bh_test.name] = pd.Series(
    perf_stats(spy_bh_test.pct_change().fillna(spy_bh_test.iloc[0] / PORT.initial_equity - 1)))
display(summary_b.T)
plot_equity({**{k: b.equity for k, b in backtests.items()}, spy_bh_test.name: spy_bh_test}, None,
            f"Experiment B: walk-forward OR-window selection on ETFs vs {BENCHMARK} buy & hold")

,days,total_return_%,CAGR_%,vol_%,sharpe,sortino,MDD_%,trades,exposure_%,hit_%,avg_R,R_tstat,profit_factor
"ML: best predicted window, skip if ≤ 0","2,386.0000",49.6737,4.3514,12.2498,0.4085,0.6720,-24.7908,"1,880.0000",53.2272,50.2128,0.0219,1.1416,1.0953
ML: best predicted window,"2,386.0000",18.0125,1.7646,12.5781,0.2014,0.3300,-32.2273,"2,069.0000",56.6220,49.0092,0.0028,0.1556,1.0357
Trailing-best window (per ETF),"2,386.0000",21.8360,2.1079,12.8682,0.2259,0.3644,-42.8452,"2,089.0000",57.5021,49.0187,0.0020,0.1119,1.0437
Static 60m (Experiment A's window),"2,386.0000",-24.9042,-2.9796,12.3507,-0.1835,-0.2788,-34.8183,"2,200.0000",58.5918,47.8636,-0.0228,-1.3835,0.9482
Buy & hold SPY,"2,386.0000",265.8297,14.6810,18.4168,0.8365,1.1644,-33.8463,NaN,NaN,NaN,NaN,NaN,NaN


## 10. Roadmap — modules not yet implemented

The previous notebook had empty placeholder cells for these. They are kept here as a backlog so they don't
show up as code that silently does nothing:

* **Double-down sizing**: scale risk on sessions the model rates as high-probability. It needs calibrated
  probabilities (e.g. isotonic calibration on walk-forward folds) before sizing can use them.
* **Volume-profile features**: prior-day value area / POC relative to the open.
* **Neural-network model**: only worth trying after a tree model shows a significant edge over the trailing-best baseline.
* **Prop-firm optimisation** (futures accounts; would need a futures data source back in scope): account rules (trailing max loss limit, daily loss limit) built into the portfolio simulator,
  plus a Monte-Carlo estimate of the probability of passing.
* **Additional validation**: rolling and randomised walk-forward, block-bootstrap confidence intervals on OOS
  Sharpe, and White's Reality Check / SPA test across the whole configuration set.